# ⚡ WaveForge 3D — Full Benchmark: CPU vs GPU vs PyMEEP

**Just click `Runtime → Run all` — everything is automated.**

This notebook:
1. Clones the WaveForge repo
2. Installs all dependencies
3. Detects GPU / CPU
4. Runs grid-scaling benchmark (32³ → 256³)
5. Runs all 10 3D physics examples
6. Loads pre-recorded PyMEEP & CPU baselines from the repo
7. Generates all comparison plots
8. Prints full summary table
9. Downloads results JSON

---
**Recommended runtime:** `Runtime → Change runtime type → T4 GPU`  
CPU-only also works (slower, but all cells complete).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Clone repo + install deps  (runs once, ~30 seconds)           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os, pathlib

REPO_URL  = 'https://github.com/shahzaibshazoo/waveforge.git'
REPO_DIR  = pathlib.Path('/content/waveforge')

if not REPO_DIR.exists():
    print('Cloning WaveForge...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already cloned — pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

# Add src to Python path
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# Ensure torch is available (Colab always has it, but just in case)
try:
    import torch
    print(f'PyTorch {torch.__version__} already available')
except ImportError:
    print('Installing PyTorch...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', '--quiet'], check=True)
    import torch

# numpy / matplotlib are always present on Colab
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('\n✅ Setup complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Detect hardware                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import platform
import torch

HAS_GPU = torch.cuda.is_available()
DEVICE  = 'cuda' if HAS_GPU else 'cpu'

print('=' * 60)
print(f'  Platform : {platform.node()} — {platform.system()} {platform.release()}')
print(f'  Python   : {platform.python_version()}')
print(f'  PyTorch  : {torch.__version__}')
if HAS_GPU:
    print(f'  CUDA     : {torch.version.cuda}')
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU[{i}]   : {props.name}  ({props.total_memory/1e9:.1f} GB)')
    GPU_NAME = torch.cuda.get_device_name(0)
else:
    print('  GPU      : NOT AVAILABLE  (running on CPU)')
    print('  ⚠  Switch to GPU: Runtime → Change runtime type → T4 GPU')
    GPU_NAME = 'CPU'
print('=' * 60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load pre-recorded baselines from repo                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import json

def load_json(path):
    with open(path) as f:
        return json.load(f)

# WaveForge CPU + PyMEEP CPU 2D scaling  (recorded on local dev machine)
cpu_scaling   = load_json('benchmarks/cpu_results.json')
# PyMEEP vs WaveForge per-scene (7 scenes)
meep_cmp      = load_json('benchmarks/meep_comparison_results.json')
# Kaggle 2×T4 GPU results
kaggle_data   = load_json('benchmarks/kaggle_gpu_results.json')
# All 10 new 3D examples on CPU
cpu_3d        = load_json('benchmarks/3d_examples_cpu_results.json')

print('Baselines loaded:')
print(f'  CPU scaling    : {len(cpu_scaling["waveforge_cpu"])} grid sizes')
print(f'  PyMEEP compare : {len(meep_cmp)} scenes')
print(f'  Kaggle T4      : {len(kaggle_data["gpu_scaling"]["cuda:0"]["rows"])} grid sizes + {len(kaggle_data["examples"])} examples')
print(f'  3D CPU examples: {len(cpu_3d)} examples')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Grid-scaling benchmark on this machine  (~2 min on T4)        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import time
import torch
from core.grid import YeeGrid
from core.fields import FieldSet
from core.boundaries import MurABC3D
from core.sources import GaussianPulse, PointSource, SourceCollection
from core.fdtd3d import FDTD3D

N_WARMUP    = 20
N_STEPS     = 100
GRID_SIZES  = [32, 48, 64, 96, 128, 192, 256, 384, 512] if HAS_GPU else [32, 48, 64, 96, 128]

def benchmark_grid_size(N, device, warmup=20, steps=100):
    DX = 1.5e-3
    grid     = YeeGrid(N, N, dx=DX, dy=DX, Nz=N, dz=DX, device=device)
    fields   = FieldSet(grid)
    boundary = MurABC3D(grid, fields.Hx, fields.Hy, fields.Hz)
    pulse    = GaussianPulse(amplitude=1.0, sigma=20 * grid.dt)
    cx = cy = cz = N // 2
    src = PointSource(pulse, cx, cy, 'Ez', k=cz, grid=grid, N_steps=warmup + steps)
    sim = FDTD3D(grid, fields, boundary, SourceCollection([src]), n_check=999999)

    with torch.no_grad():
        sim.run(warmup)
    if device == 'cuda': torch.cuda.synchronize()

    t0 = time.perf_counter()
    with torch.no_grad():
        sim.run(steps)
    if device == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    mcells_s = steps * N**3 / elapsed / 1e6
    ms_step  = elapsed / steps * 1e3
    return round(mcells_s, 1), round(ms_step, 3)

print(f'Running grid-scaling on {DEVICE} (warmup={N_WARMUP}, timed={N_STEPS} steps each)...')
print(f'{"N":>6}  {"Mcells/s":>10}  {"ms/step":>10}')
print('-' * 30)

this_session_scaling = []
for N in GRID_SIZES:
    try:
        mc, ms = benchmark_grid_size(N, DEVICE, N_WARMUP, N_STEPS)
        this_session_scaling.append({'N': N, 'mcells_s': mc, 'ms_step': ms})
        print(f'{N:6d}³  {mc:10.1f}  {ms:10.3f}')
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        print(f'{N:6d}³  OOM — stopping here')
        break

print('\n✅ Grid-scaling benchmark done.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Run all 10 3D examples on this machine  (~5 min on T4)        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import re, subprocess, time, os

# make sure output dir exists
os.makedirs('examples/output', exist_ok=True)

this_session_examples = []

for entry in cpu_3d:
    fname = entry['file']
    path  = f'examples/3d/{fname}'
    print(f'  {fname}...', end=' ', flush=True)
    t0 = time.time()
    try:
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
        res = subprocess.run([sys.executable, path],
                             capture_output=True, text=True,
                             timeout=360, env=env)
        elapsed = time.time() - t0
        out  = res.stdout + res.stderr
        m    = re.search(r'WAVEFORGE_BENCH:\s*([\d.]+)', out)
        g    = re.search(r'Grid:\s*(\d+)x(\d+)x(\d+)', out)
        mc   = float(m.group(1)) if m else 0.0
        grid = f'{g.group(1)}x{g.group(2)}x{g.group(3)}' if g else entry['grid']
        ok   = res.returncode == 0 and mc > 0
        status = 'PASS' if ok else 'FAIL'
        this_session_examples.append({
            'file': fname, 'status': status,
            'time_s': round(elapsed, 1),
            'mcells_s': mc, 'grid': grid
        })
        print(f'{status} | {elapsed:.1f}s | {mc:.1f} Mcells/s | {grid}')
        if not ok:
            # print last 3 lines of output for debugging
            for line in out.strip().split('\n')[-3:]:
                print(f'    {line}')
    except subprocess.TimeoutExpired:
        this_session_examples.append({
            'file': fname, 'status': 'TIMEOUT',
            'time_s': 360, 'mcells_s': 0, 'grid': entry['grid']
        })
        print('TIMEOUT (>360s)')

n_pass = sum(1 for r in this_session_examples if r['status'] == 'PASS')
print(f'\n✅ Examples done: {n_pass}/{len(this_session_examples)} passed')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Save results to JSON                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import datetime

session_results = {
    'meta': {
        'date'     : datetime.datetime.now().isoformat(),
        'device'   : GPU_NAME,
        'has_gpu'  : HAS_GPU,
        'vram_gb'  : round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if HAS_GPU else 0,
        'torch'    : torch.__version__,
        'cuda'     : torch.version.cuda if HAS_GPU else 'N/A',
        'python'   : platform.python_version(),
        'warmup'   : N_WARMUP,
        'steps'    : N_STEPS,
    },
    'grid_scaling' : this_session_scaling,
    'examples'     : this_session_examples,
}

safe_name = GPU_NAME.replace(' ', '_').replace('/', '-')
out_path  = f'benchmarks/colab_{safe_name}_results.json'
with open(out_path, 'w') as f:
    json.dump(session_results, f, indent=2)

print(f'Results saved → {out_path}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Plot 1: Grid scaling throughput + speedup                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('WaveForge 3D — Grid Scaling Benchmark', fontsize=14, fontweight='bold')

# ── Left: Throughput ──────────────────────────────────────────────────────
ax = axes[0]

cpu_N   = [r['N'] for r in cpu_scaling['waveforge_cpu']]
cpu_mc  = [r['mcells_s'] for r in cpu_scaling['waveforge_cpu']]
ax.plot(cpu_N, cpu_mc, 'o--', color='royalblue', lw=2, label='WaveForge CPU (baseline)')

meep_N  = [r['N'] for r in cpu_scaling['meep_cpu']]
meep_mc = [r['mcells_s'] for r in cpu_scaling['meep_cpu']]
ax.plot(meep_N, meep_mc, 'x:', color='gray', lw=1.5, label='PyMEEP CPU (reference)')

t4_rows = kaggle_data['gpu_scaling']['cuda:0']['rows']
ax.plot([r['N'] for r in t4_rows], [r['mcells_s'] for r in t4_rows],
        's-', color='darkorange', lw=2, label='Kaggle Tesla T4')

if this_session_scaling:
    label = f'This session ({GPU_NAME.split()[0] if HAS_GPU else "CPU"})'
    ax.plot([r['N'] for r in this_session_scaling],
            [r['mcells_s'] for r in this_session_scaling],
            '^-', color='crimson', lw=2.5, label=label)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Grid size N  (N³ cells)', fontsize=11)
ax.set_ylabel('Throughput (Mcells/s)', fontsize=11)
ax.set_title('Throughput vs Grid Size')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}³'))

# ── Right: Speedup vs WaveForge CPU ──────────────────────────────────────
ax2 = axes[1]
cpu_dict = {r['N']: r['mcells_s'] for r in cpu_scaling['waveforge_cpu']}

def speedup_series(rows):
    ns  = [r['N'] for r in rows if r['N'] in cpu_dict]
    spu = [r['mcells_s'] / cpu_dict[r['N']] for r in rows if r['N'] in cpu_dict]
    return ns, spu

ns, spu = speedup_series(t4_rows)
ax2.plot(ns, spu, 's-', color='darkorange', lw=2, label='Kaggle T4 vs CPU')

if this_session_scaling and HAS_GPU:
    ns2, spu2 = speedup_series(this_session_scaling)
    ax2.plot(ns2, spu2, '^-', color='crimson', lw=2.5,
             label=f'{GPU_NAME.split()[0]} vs CPU')

ax2.axhline(1, color='royalblue', ls='--', lw=1, label='CPU baseline (1×)')
ax2.set_xscale('log', base=2)
ax2.set_xlabel('Grid size N  (N³ cells)', fontsize=11)
ax2.set_ylabel('GPU / CPU speedup  (×)', fontsize=11)
ax2.set_title('GPU Speedup over WaveForge CPU')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, which='both')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)}³'))

plt.tight_layout()
os.makedirs('docs/assets', exist_ok=True)
plt.savefig('docs/assets/colab_scaling_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/colab_scaling_benchmark.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Plot 2: All 10 examples — CPU vs Kaggle T4 vs This GPU        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle('WaveForge 3D — All 10 Examples Benchmark', fontsize=14, fontweight='bold')

short_names = [r['file'].replace('3d_', '').replace('.py', '') for r in cpu_3d]
cpu_mc_ex   = [r['mcells_s'] for r in cpu_3d]
this_mc_ex  = [r['mcells_s'] for r in this_session_examples]

# Kaggle T4 example results (stored in kaggle_data)
kaggle_ex_mc = [r['mcells_s'] for r in kaggle_data['examples']]
# Align lengths
n = len(short_names)
kaggle_ex_mc = (kaggle_ex_mc + [0]*n)[:n]

x     = np.arange(n)
w     = 0.26
label_gpu = f'This GPU ({GPU_NAME.split()[0]})' if HAS_GPU else 'This CPU'

# ── Throughput ──
ax = axes[0]
b1 = ax.bar(x - w,     cpu_mc_ex,    w, label='WaveForge CPU', color='royalblue', alpha=0.85)
b2 = ax.bar(x,         kaggle_ex_mc, w, label='Kaggle T4',     color='darkorange', alpha=0.85)
b3 = ax.bar(x + w,     this_mc_ex,   w, label=label_gpu,       color='crimson', alpha=0.85)

# Annotate speedup on top of "this GPU" bar
for i, (c, g) in enumerate(zip(cpu_mc_ex, this_mc_ex)):
    if c > 0 and g > 0:
        ax.text(x[i]+w, g + 0.4, f'{g/c:.1f}×', ha='center', va='bottom', fontsize=7, color='darkred', fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(short_names, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Throughput (Mcells/s)', fontsize=11)
ax.set_title('Throughput per Example')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── Wall time ──
ax2 = axes[1]
cpu_t_ex   = [r['time_s'] for r in cpu_3d]
kaggle_t   = [r.get('time_s', 0) for r in kaggle_data['examples']]
kaggle_t   = (kaggle_t + [0]*n)[:n]
this_t_ex  = [r['time_s'] for r in this_session_examples]

ax2.bar(x - w, cpu_t_ex,  w, label='WaveForge CPU', color='royalblue', alpha=0.85)
ax2.bar(x,     kaggle_t,  w, label='Kaggle T4',     color='darkorange', alpha=0.85)
ax2.bar(x + w, this_t_ex, w, label=label_gpu,       color='crimson', alpha=0.85)

ax2.set_xticks(x); ax2.set_xticklabels(short_names, rotation=40, ha='right', fontsize=8)
ax2.set_ylabel('Wall time (s)', fontsize=11)
ax2.set_title('Runtime per Example')
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/colab_examples_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/colab_examples_benchmark.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Plot 3: PyMEEP CPU vs WaveForge CPU vs WaveForge GPU          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('WaveForge vs PyMEEP — Head-to-Head Comparison', fontsize=14, fontweight='bold')

scenes        = list(meep_cmp.keys())
meep_vals     = [meep_cmp[s]['meep']['mcells_s']      for s in scenes]
wf_cpu_vals   = [meep_cmp[s]['waveforge']['mcells_s'] for s in scenes]

# Match WaveForge GPU per scene by index (7 scenes → first 7 from this session)
wf_gpu_vals = []
for i in range(len(scenes)):
    if i < len(this_session_examples) and this_session_examples[i]['mcells_s'] > 0:
        wf_gpu_vals.append(this_session_examples[i]['mcells_s'])
    elif i < len(kaggle_data['examples']):
        wf_gpu_vals.append(kaggle_data['examples'][i]['mcells_s'])
    else:
        wf_gpu_vals.append(0)

x  = np.arange(len(scenes))
w  = 0.26
ax = axes[0]
ax.bar(x - w, meep_vals,   w, label='PyMEEP CPU',    color='slategray', alpha=0.85)
ax.bar(x,     wf_cpu_vals, w, label='WaveForge CPU', color='royalblue', alpha=0.85)
ax.bar(x + w, wf_gpu_vals, w, label=label_gpu,       color='crimson', alpha=0.85)

# Annotate GPU/Meep speedup
for i, (m, g) in enumerate(zip(meep_vals, wf_gpu_vals)):
    if m > 0 and g > 0:
        ax.text(x[i]+w, g+0.5, f'{g/m:.0f}×\nMeep', ha='center', fontsize=7, color='darkred', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in scenes], fontsize=8)
ax.set_ylabel('Throughput (Mcells/s)', fontsize=11)
ax.set_title('Throughput: PyMEEP vs WaveForge')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── Speedup bars ──
ax2 = axes[1]
gpu_vs_meep = [g/m if m > 0 else 0 for g, m in zip(wf_gpu_vals, meep_vals)]
wf_vs_meep  = [c/m if m > 0 else 0 for c, m in zip(wf_cpu_vals, meep_vals)]

ax2.bar(x - w/2, wf_vs_meep,  w, label='WaveForge CPU / PyMEEP', color='royalblue', alpha=0.85)
ax2.bar(x + w/2, gpu_vs_meep, w, label=f'{label_gpu} / PyMEEP',  color='crimson', alpha=0.85)
ax2.axhline(1, color='gray', ls='--', lw=1, label='PyMEEP baseline')

for i, (c, g) in enumerate(zip(wf_vs_meep, gpu_vs_meep)):
    ax2.text(x[i]-w/2, c+0.05, f'{c:.1f}×', ha='center', fontsize=7)
    ax2.text(x[i]+w/2, g+0.05, f'{g:.0f}×', ha='center', fontsize=7, color='darkred', fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels([s.replace('_', '\n') for s in scenes], fontsize=8)
ax2.set_ylabel('Speedup vs PyMEEP (×)', fontsize=11)
ax2.set_title('Speedup over PyMEEP CPU')
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('docs/assets/colab_meep_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/colab_meep_comparison.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Full Summary Table                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print()
print('╔' + '═'*72 + '╗')
print(f'║  WAVEFORGE 3D BENCHMARK — {GPU_NAME:<44} ║')
print('╠' + '═'*72 + '╣')

# ── Grid scaling ──
print(f'║  Grid Scaling  ({N_WARMUP} warmup + {N_STEPS} timed steps per size)                          ║')
print(f'║  {"N":>6}  {"WF-CPU (Mc/s)":>14}  {"Kaggle T4 (Mc/s)":>16}  {"This GPU (Mc/s)":>16}  {"Speedup":>6}  ║')
print('║' + '─'*72 + '║')

cpu_d  = {r['N']: r['mcells_s'] for r in cpu_scaling['waveforge_cpu']}
t4_d   = {r['N']: r['mcells_s'] for r in kaggle_data['gpu_scaling']['cuda:0']['rows']}
this_d = {r['N']: r['mcells_s'] for r in this_session_scaling}

for N in sorted(cpu_d | t4_d | this_d):
    c = cpu_d.get(N, 0);  t = t4_d.get(N, 0);  g = this_d.get(N, 0)
    spd = f'{g/c:.1f}×' if c > 0 and g > 0 else ('-' if c == 0 else '')
    print(f'║  {N:6d}³  {c:14.1f}  {t:16.1f}  {g:16.1f}  {spd:>6}   ║')

if this_d:
    peak_n  = max(this_d, key=this_d.get)
    peak_mc = this_d[peak_n]
    print('║' + '─'*72 + '║')
    print(f'║  Peak: {peak_mc:.1f} Mcells/s at {peak_n}³{" "*(52-len(str(peak_n)))  }║')

# ── Examples ──
print('╠' + '═'*72 + '╣')
print(f'║  All 10 3D Examples                                                      ║')
print(f'║  {"Example":<30}  {"CPU":>8}  {"T4":>8}  {"This GPU":>10}  {"Speedup":>8}  ║')
print('║' + '─'*72 + '║')

total_cpu_t = total_this_t = 0
for c_entry, g_entry, k_entry in zip(cpu_3d, this_session_examples, kaggle_data['examples']):
    name = c_entry['file'].replace('3d_', '').replace('.py', '')
    c_mc = c_entry['mcells_s']
    k_mc = k_entry['mcells_s']
    g_mc = g_entry['mcells_s']
    spd  = f'{g_mc/c_mc:.1f}×' if c_mc > 0 and g_mc > 0 else '-'
    print(f'║  {name:<30}  {c_mc:>8.1f}  {k_mc:>8.1f}  {g_mc:>10.1f}  {spd:>8}  ║')
    total_cpu_t  += c_entry['time_s']
    total_this_t += g_entry['time_s']

print('║' + '─'*72 + '║')
time_spd = f'{total_cpu_t/total_this_t:.1f}×' if total_this_t > 0 else '-'
print(f'║  {"Total wall time":<30}  {total_cpu_t:>7.0f}s  {" ":>8}  {total_this_t:>9.0f}s  {time_spd:>8}  ║')

# ── PyMEEP comparison ──
print('╠' + '═'*72 + '╣')
print(f'║  vs PyMEEP CPU  (pre-recorded 7-scene comparison)                        ║')
print(f'║  {"Scene":<25}  {"PyMEEP (Mc/s)":>13}  {"WF-CPU (Mc/s)":>14}  {"WF-GPU (Mc/s)":>14}  {"GPU/Meep":>8} ║')
print('║' + '─'*72 + '║')
for i, s in enumerate(scenes):
    m = meep_cmp[s]['meep']['mcells_s']
    c = meep_cmp[s]['waveforge']['mcells_s']
    g = wf_gpu_vals[i]
    spd = f'{g/m:.0f}×' if m > 0 and g > 0 else '-'
    print(f'║  {s:<25}  {m:>13.1f}  {c:>14.1f}  {g:>14.1f}  {spd:>8} ║')

print('╚' + '═'*72 + '╝')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Download results + plots to local machine                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
try:
    from google.colab import files
    print('Downloading results...')
    files.download(out_path)
    for img in ['docs/assets/colab_scaling_benchmark.png',
                'docs/assets/colab_examples_benchmark.png',
                'docs/assets/colab_meep_comparison.png']:
        if os.path.exists(img):
            files.download(img)
    print('✅ All files downloaded.')
except ImportError:
    print(f'Not running on Colab — files saved locally at:')
    print(f'  {out_path}')
    for img in ['docs/assets/colab_scaling_benchmark.png',
                'docs/assets/colab_examples_benchmark.png',
                'docs/assets/colab_meep_comparison.png']:
        if os.path.exists(img): print(f'  {img}')

print()
print('🏁  All done! Benchmark complete.')